# Global Electronics Retailer — Sales & Customer Analytics

**Portfolio project | Python · Pandas · NumPy · Matplotlib · Seaborn · Jupyter Notebook**

## Executive overview

This project analyses a multi-table retail dataset to evaluate sales performance, profitability, customer purchasing behaviour, channel and store performance, and delivery outcomes. The workflow covers data-quality validation, cleaning, relational joins, feature engineering, exploratory analysis, customer RFM segmentation, visualizations, and exportable analytical outputs.

**Business objective:** Convert raw retail data into reliable, decision-ready evidence that can help commercial and operations teams understand what drives revenue and profit, identify improvement opportunities, and prioritize customer engagement.


## 1. Business problem

Retail teams often review sales, product, customer, and store data separately. This makes it difficult to form a consistent view of business performance, distinguish revenue growth from profitability, identify high-value customer groups, and connect delivery performance with commercial outcomes.

This analysis creates a consolidated view of transactions and addresses four business needs:

- **Commercial performance:** Understand revenue, profit, margin, and changes over time.
- **Product and category strategy:** Identify leading products and categories, including high-revenue products with comparatively low margins.
- **Customer value:** Measure repeat purchasing, revenue concentration, and RFM-based customer segments.
- **Operational performance:** Compare channels, store productivity, and delivery duration while validating the quality of the underlying data.


## 2. Business questions

1. How do revenue and profit change over time?
2. Which categories contribute most to revenue and profit?
3. Which categories have comparatively low profit margins?
4. Which products lead in revenue, and which combine high revenue with low margin?
5. How concentrated is revenue among the highest-value customers?
6. What are the revenue, profit, order, and customer differences by sales channel?
7. Which stores generate the strongest revenue and revenue per square metre?
8. What does delivery duration indicate about fulfilment performance?
9. Which RFM customer segments contribute the most revenue and profit?
10. What data-quality issues must be resolved or monitored before decisions are made?


## 3. Data and analytical approach

### Source tables

- **Customers:** Customer attributes and geography
- **Products:** Product, brand, category, unit price, and unit cost
- **Sales:** Order-line transactions, dates, product/customer/store keys, and quantity
- **Stores:** Store geography, size, and opening date
- **Exchange Rates:** Currency conversion reference data

### Workflow

1. Load the source tables and inspect structure and quality.
2. Validate primary, composite, and foreign-key relationships, date fields, and numeric business rules.
3. Standardize text, date, numeric, and currency-formatted fields.
4. Merge sales with product, customer, and store attributes while validating join cardinality.
5. Engineer financial, calendar, delivery, customer-age, channel, and RFM features.
6. Analyse business questions and produce charts and executive findings.
7. Export analytical datasets and the data-quality report.

**Financial definitions:** Revenue = quantity × unit selling price; cost = quantity × unit cost; profit = revenue − cost; profit margin = profit ÷ revenue. The source price and cost fields are denominated in USD.


## 4. Tools and techniques

- **Python:** Reusable analysis workflow and business-rule implementation
- **Pandas:** Data cleaning, validation, joins, aggregation, and segmentation
- **NumPy:** Numeric transformations and conditional feature engineering
- **Matplotlib and Seaborn:** Business-focused visualizations
- **Jupyter Notebook:** Reproducible analysis and reporting

The analysis emphasizes data integrity and interpretable descriptive analytics. RFM scoring organizes customers by recency, frequency, and monetary value; it is not presented as a predictive model.


## 5. Data loading and quality validation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from pandas.api.types import (
    is_string_dtype,
    is_numeric_dtype
)

pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)
pd.set_option('display.max_colwidth',30)
pd.set_option('display.width',None)
sns.set_theme(style='whitegrid')

In [ ]:
Base_path = Path('../../Project/Global+Electronics+Retailer')

customers_path = Base_path / 'Customers.csv'
products_path = Base_path / 'Products.csv'
sales_path = Base_path / 'Sales.csv'
stores_path = Base_path / 'Stores.csv'
exchange_path = Base_path / 'Exchange_Rates.csv'

In [ ]:
def read_csv_with_encoding(path, encodings=("utf-8", "utf-8-sig", "cp1252", "latin1")):
    last_error = None

    for encoding in encodings:
        try:
            data = pd.read_csv(path, encoding=encoding)
            print(f"Loaded {Path(path).name} using encoding: {encoding}")
            return data
        except UnicodeDecodeError as exc:
            last_error = exc

    raise UnicodeDecodeError(
        last_error.encoding,
        last_error.object,
        last_error.start,
        last_error.end,
        f"Could not decode {path} using encodings: {encodings}"
    )

customers = read_csv_with_encoding(customers_path)
products = read_csv_with_encoding(products_path)
sales = read_csv_with_encoding(sales_path)
stores = read_csv_with_encoding(stores_path)
exchange_rates = read_csv_with_encoding(exchange_path)

In [ ]:
datasets = {
    'Customers': customers,
    'Products': products,
    'Sales': sales,
    'Stores': stores,
    'Exchange_Rates': exchange_rates
}

In [ ]:
primary_keys = {
    'Customers': 'CustomerKey',
    'Products': 'ProductKey',
    'Stores': 'StoreKey'
}

In [ ]:
composite_keys = {
    'Sales': ['Order Number', 'Line Item']
}

In [ ]:
foreign_keys = [

    {
        'child_table': 'Sales',
        'child_key': 'CustomerKey',
        'parent_table': 'Customers',
        'parent_key': 'CustomerKey'
    },

    {
        'child_table': 'Sales',
        'child_key': 'ProductKey',
        'parent_table': 'Products',
        'parent_key': 'ProductKey'
    },

    {
        'child_table': 'Sales',
        'child_key': 'StoreKey',
        'parent_table': 'Stores',
        'parent_key': 'StoreKey'
    }
]

In [ ]:
date_columns = [

    {
        'table': 'Customers',
        'column': 'Birthday'
    },

    {
        'table': 'Sales',
        'column': 'Order Date'
    },

    {
        'table': 'Sales',
        'column': 'Delivery Date'
    },

    {
        'table': 'Stores',
        'column': 'Open Date'
    },

    {
        'table': 'Exchange_Rates',
        'column': 'Exchange'
    }
]

In [ ]:
numeric_rules = [

    {
        'table': 'Sales',
        'column': 'Quantity',
        'rule_name': 'Quantity must be greater than zero',
        'condition': lambda s: pd.to_numeric(
            s,
            errors='coerce'
        ) <= 0
    },

    {
        'table': 'Products',
        'column': 'Unit Cost USD',
        'rule_name': 'Unit cost cannot be negative',
        'condition': lambda s: pd.to_numeric(
            s,
            errors='coerce'
        ) < 0
    },

    {
        'table': 'Products',
        'column': 'Unit Price USD',
        'rule_name': 'Unit price cannot be negative',
        'condition': lambda s: pd.to_numeric(
            s,
            errors='coerce'
        ) < 0
    },

    {
        'table': 'Stores',
        'column': 'Square Meters',
        'rule_name': 'Store area cannot be negative',
        'condition': lambda s: pd.to_numeric(
            s,
            errors='coerce'
        ) < 0
    },

    {
        'table': 'Exchange_Rates',
        'column': 'Exchange',
        'rule_name': 'Exchange rate must be greater than zero',
        'condition': lambda s: pd.to_numeric(
            s,
            errors='coerce'
        ) <= 0
    }
]

In [ ]:
business_rules = [
    
    {
        'name': 'Delivery Date cannot be before Order Date',
        'table': 'Sales',
        'columns': ['Order Date', 'Delivery Date']
    }
]

In [ ]:
def validate_quality_config(
    df_dict,
    primary_keys=None,
    composite_keys=None,
    foreign_keys=None,
    date_columns=None,
    numeric_rules=None
):

    errors = []

    primary_keys = primary_keys or {}
    composite_keys = composite_keys or {}
    foreign_keys = foreign_keys or []
    date_columns = date_columns or []
    numeric_rules = numeric_rules or []

    for table, column in primary_keys.items():

        if table not in df_dict:
            errors.append(
                f"Primary-key table '{table}' does not exist."
            )
            continue

        if column not in df_dict[table].columns:
            errors.append(
                f"Primary-key column '{column}' "
                f"not found in '{table}'."
            )

    for table, columns in composite_keys.items():

        if table not in df_dict:
            errors.append(
                f"Composite-key table '{table}' does not exist."
            )
            continue

        for column in columns:

            if column not in df_dict[table].columns:
                errors.append(
                    f"Composite-key column '{column}' "
                    f"not found in '{table}'."
                )

    for fk in foreign_keys:

        child_table = fk['child_table']
        child_key = fk['child_key']

        parent_table = fk['parent_table']
        parent_key = fk['parent_key']

        if child_table not in df_dict:

            errors.append(
                f"Foreign-key child table "
                f"'{child_table}' does not exist."
            )

        elif child_key not in df_dict[child_table].columns:

            errors.append(
                f"Foreign-key column '{child_key}' "
                f"not found in '{child_table}'."
            )

        if parent_table not in df_dict:

            errors.append(
                f"Foreign-key parent table "
                f"'{parent_table}' does not exist."
            )

        elif parent_key not in df_dict[parent_table].columns:

            errors.append(
                f"Parent key '{parent_key}' "
                f"not found in '{parent_table}'."
            )

    for rule in date_columns:

        table = rule['table']
        column = rule['column']

        if table not in df_dict:

            errors.append(
                f"Date-rule table '{table}' does not exist."
            )

        elif column not in df_dict[table].columns:

            errors.append(
                f"Date column '{column}' "
                f"not found in '{table}'."
            )

    for rule in numeric_rules:

        table = rule['table']
        column = rule['column']

        if table not in df_dict:

            errors.append(
                f"Numeric-rule table '{table}' does not exist."
            )

        elif column not in df_dict[table].columns:

            errors.append(
                f"Numeric-rule column '{column}' "
                f"not found in '{table}'."
            )

    if errors:

        error_message = "\n".join(
            f"{i}. {error}"
            for i, error in enumerate(errors, start=1)
        )

        raise ValueError(
            "Data quality configuration contains errors:\n\n"
            + error_message
        )

    return True

In [ ]:
def data_quality_report_generation(
    df_dict,
    primary_keys=None,
    composite_keys=None,
    foreign_keys=None,
    date_columns=None,
    numeric_rules=None
):

    if not isinstance(df_dict, dict):
        raise TypeError(
            "df_dict must be a dictionary of DataFrames."
        )

    if not df_dict:
        raise ValueError(
            "df_dict is empty. No datasets available."
        )

    validate_quality_config(
        df_dict=df_dict,
        primary_keys=primary_keys,
        composite_keys=composite_keys,
        foreign_keys=foreign_keys,
        date_columns=date_columns,
        numeric_rules=numeric_rules
    )

    report_list = []

    for table_name, df in df_dict.items():

        if not isinstance(df, pd.DataFrame):
            raise TypeError(
                f"'{table_name}' is not a pandas DataFrame."
            )

        total_rows = len(df)
        total_columns = len(df.columns)

        duplicate_row_count = int(
            df.duplicated().sum()
        )

        for column in df.columns:

            series = df[column]

            missing_count = int(
                series.isna().sum()
            )

            missing_pct = (
                round(
                    missing_count / total_rows * 100,
                    2
                )
                if total_rows > 0
                else 0
            )

            unique_count = int(
                series.nunique(dropna=True)
            )

            repeated_value_count = int(
                series.duplicated().sum()
            )

            blank_count = 0

            if is_string_dtype(series):

                blank_count = int(
                    series
                    .fillna('')
                    .astype(str)
                    .str.strip()
                    .eq('')
                    .sum()
                )

            report_list.append({

                'Check Type': 'Column Quality',

                'Table Name': table_name,

                'Column Name': column,

                'Data Type': str(series.dtype),

                'Total Rows': total_rows,

                'Total Columns': total_columns,

                'Missing Count': missing_count,

                'Missing %': missing_pct,

                'Unique Count': unique_count,

                'Repeated Value Count':
                    repeated_value_count,

                'Duplicate Row Count':
                    duplicate_row_count,

                'Blank Count':
                    blank_count,

                'Primary Key': '',

                'Foreign Key': '',

                'Parent Table': '',

                'Parent Key': '',

                'Invalid Count': '',

                'Status': ''
            })

    for table, column in (
        primary_keys or {}
    ).items():

        df = df_dict[table]

        missing_pk = int(
            df[column].isna().sum()
        )

        duplicate_pk = int(
            df[column].duplicated().sum()
        )

        invalid_count = (
            missing_pk + duplicate_pk
        )

        status = (
            'PASS'
            if invalid_count == 0
            else 'FAIL'
        )

        report_list.append({

            'Check Type': 'Primary Key',

            'Table Name': table,

            'Column Name': column,

            'Data Type':
                str(df[column].dtype),

            'Total Rows': len(df),

            'Total Columns': len(df.columns),

            'Missing Count': missing_pk,

            'Missing %':
                round(
                    missing_pk / len(df) * 100,
                    2
                )
                if len(df) > 0 else 0,

            'Unique Count':
                df[column].nunique(
                    dropna=True
                ),

            'Repeated Value Count':
                duplicate_pk,

            'Duplicate Row Count':
                int(df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': 'YES',

            'Foreign Key': '',

            'Parent Table': '',

            'Parent Key': '',

            'Invalid Count': invalid_count,

            'Status': status
        })

    for table, columns in (
        composite_keys or {}
    ).items():

        df = df_dict[table]

        missing_key = int(
            df[columns]
            .isna()
            .any(axis=1)
            .sum()
        )

        duplicate_key = int(
            df.duplicated(
                subset=columns
            ).sum()
        )

        invalid_count = (
            missing_key + duplicate_key
        )

        status = (
            'PASS'
            if invalid_count == 0
            else 'FAIL'
        )

        report_list.append({

            'Check Type': 'Composite Key',

            'Table Name': table,

            'Column Name':
                ' + '.join(columns),

            'Data Type': 'Multiple',

            'Total Rows': len(df),

            'Total Columns': len(df.columns),

            'Missing Count': missing_key,

            'Missing %':
                round(
                    missing_key / len(df) * 100,
                    2
                )
                if len(df) > 0 else 0,

            'Unique Count':
                df[columns]
                .drop_duplicates()
                .shape[0],

            'Repeated Value Count':
                duplicate_key,

            'Duplicate Row Count':
                int(df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': 'COMPOSITE',

            'Foreign Key': '',

            'Parent Table': '',

            'Parent Key': '',

            'Invalid Count':
                invalid_count,

            'Status': status
        })

    for fk in foreign_keys or []:

        child_table = fk['child_table']
        child_key = fk['child_key']

        parent_table = fk['parent_table']
        parent_key = fk['parent_key']

        child_df = df_dict[child_table]
        parent_df = df_dict[parent_table]

        parent_values = set(
            parent_df[parent_key]
            .dropna()
            .unique()
        )

        orphan_count = int(
            (
                child_df[child_key].notna()
                &
                ~child_df[child_key].isin(
                    parent_values
                )
            ).sum()
        )

        missing_fk = int(
            child_df[child_key].isna().sum()
        )

        invalid_count = (
            orphan_count + missing_fk
        )

        status = (
            'PASS'
            if invalid_count == 0
            else 'FAIL'
        )

        report_list.append({

            'Check Type': 'Foreign Key',

            'Table Name': child_table,

            'Column Name': child_key,

            'Data Type':
                str(child_df[child_key].dtype),

            'Total Rows': len(child_df),

            'Total Columns':
                len(child_df.columns),

            'Missing Count': missing_fk,

            'Missing %':
                round(
                    missing_fk / len(child_df) * 100,
                    2
                )
                if len(child_df) > 0 else 0,

            'Unique Count':
                child_df[child_key]
                .nunique(dropna=True),

            'Repeated Value Count':
                int(
                    child_df[child_key]
                    .duplicated()
                    .sum()
                ),

            'Duplicate Row Count':
                int(child_df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': '',

            'Foreign Key': 'YES',

            'Parent Table': parent_table,

            'Parent Key': parent_key,

            'Invalid Count': invalid_count,

            'Status': status
        })

    for rule in date_columns or []:

        table = rule['table']
        column = rule['column']

        df = df_dict[table]

        converted = pd.to_datetime(
            df[column],
            errors='coerce'
        )

        invalid_date_count = int(
            (
                df[column].notna()
                &
                converted.isna()
            ).sum()
        )

        future_date_count = int(
            (
                converted > pd.Timestamp.today()
            ).sum()
        )

        invalid_count = (
            invalid_date_count
            +
            future_date_count
        )

        status = (
            'PASS'
            if invalid_count == 0
            else 'FAIL'
        )

        report_list.append({

            'Check Type': 'Date Validation',

            'Table Name': table,

            'Column Name': column,

            'Data Type':
                str(df[column].dtype),

            'Total Rows': len(df),

            'Total Columns': len(df.columns),

            'Missing Count':
                int(df[column].isna().sum()),

            'Missing %':
                round(
                    df[column].isna().mean() * 100,
                    2
                ),

            'Unique Count':
                int(
                    df[column].nunique(
                        dropna=True
                    )
                ),

            'Repeated Value Count':
                int(
                    df[column]
                    .duplicated()
                    .sum()
                ),

            'Duplicate Row Count':
                int(df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': '',

            'Foreign Key': '',

            'Parent Table': '',

            'Parent Key': '',

            'Invalid Count': invalid_count,

            'Status': status
        })

    for rule in numeric_rules or []:

        table = rule['table']
        column = rule['column']

        df = df_dict[table]

        original = df[column]

        numeric_source = original

        if (
            table == 'Products'
            and column in ['Unit Price USD', 'Unit Cost USD']
        ):
            numeric_source = (
                original
                .astype('string')
                .str.replace('$', '', regex=False)
                .str.replace(',', '', regex=False)
                .str.strip()
            )

        numeric_values = pd.to_numeric(
            numeric_source,
            errors='coerce'
        )

        invalid_numeric_mask = (
            original.notna()
            &
            numeric_values.isna()
        )

        invalid_numeric_count = int(
            invalid_numeric_mask.sum()
        )

        business_rule_mask = rule['condition'](
            numeric_values
        )

        business_rule_count = int(
            business_rule_mask
            .fillna(False)
            .sum()
        )

        total_invalid_count = (
            invalid_numeric_count
            +
            business_rule_count
        )

        status = (
            'PASS'
            if total_invalid_count == 0
            else 'FAIL'
        )

        report_list.append({

            'Check Type':
                'Numeric Validation',

            'Table Name': table,

            'Column Name': column,

            'Data Type':
                str(original.dtype),

            'Total Rows': len(df),

            'Total Columns':
                len(df.columns),

            'Missing Count':
                int(original.isna().sum()),

            'Missing %':
                round(
                    original.isna().mean() * 100,
                    2
                ),

            'Unique Count':
                int(
                    original.nunique(
                        dropna=True
                    )
                ),

            'Repeated Value Count':
                int(original.duplicated().sum()),

            'Duplicate Row Count':
                int(df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': '',

            'Foreign Key': '',

            'Parent Table': '',

            'Parent Key': '',

            'Invalid Count':
                invalid_numeric_count,

            'Status':
                'PASS'
                if invalid_numeric_count == 0
                else 'FAIL'
        })

        report_list.append({

            'Check Type':
                'Business Rule',

            'Table Name': table,

            'Column Name':
                column,

            'Data Type':
                str(original.dtype),

            'Total Rows': len(df),

            'Total Columns':
                len(df.columns),

            'Missing Count':
                int(original.isna().sum()),

            'Missing %':
                round(
                    original.isna().mean() * 100,
                    2
                ),

            'Unique Count':
                int(
                    original.nunique(
                        dropna=True
                    )
                ),

            'Repeated Value Count':
                int(original.duplicated().sum()),

            'Duplicate Row Count':
                int(df.duplicated().sum()),

            'Blank Count': '',

            'Primary Key': '',

            'Foreign Key': '',

            'Parent Table': '',

            'Parent Key': '',

            'Invalid Count':
                business_rule_count,

            'Status':
                'PASS'
                if business_rule_count == 0
                else 'FAIL'
        })

    quality_df = pd.DataFrame(
        report_list
    )

    return quality_df

In [ ]:
data_quality = data_quality_report_generation(
    df_dict=datasets,
    primary_keys=primary_keys,
    composite_keys=composite_keys,
    foreign_keys=foreign_keys,
    date_columns=date_columns,
    numeric_rules=numeric_rules
)

display(data_quality)

In [ ]:
failed_checks = data_quality[
    data_quality['Status'] == 'FAIL'
]

display(failed_checks)

In [ ]:
quality_summary = (
    data_quality[
        data_quality['Status'].isin(['PASS', 'FAIL'])
    ]
    .groupby('Status')
    .size()
    .reset_index(name='Check Count')
)

display(quality_summary)

In [ ]:
data_quality.to_csv(
    'data_quality_report.csv',
    index=False
)

In [ ]:
sales['Order Date'] = pd.to_datetime(
    sales['Order Date'],
    errors='coerce'
)

sales['Delivery Date'] = pd.to_datetime(
    sales['Delivery Date'],
    errors='coerce'
)

invalid_delivery_dates = (
    sales['Delivery Date'].notna()
    &
    sales['Order Date'].notna()
    &
    (
        sales['Delivery Date']
        < sales['Order Date']
    )
)

print(
    "Invalid Delivery Date records:",
    invalid_delivery_dates.sum()
)

## 6. Data cleaning and preparation

In [ ]:
customers_clean = customers.copy()
products_clean = products.copy()
sales_clean = sales.copy()
stores_clean = stores.copy()
exchange_rates_clean = exchange_rates.copy()

In [ ]:
for df in [
    customers_clean,
    products_clean,
    sales_clean,
    stores_clean,
    exchange_rates_clean
]:
    
    df.columns = (
        df.columns
        .str.strip()
    )

In [ ]:
for name, df in {
    'Customers': customers_clean,
    'Products': products_clean,
    'Sales': sales_clean,
    'Stores': stores_clean,
    'Exchange Rates': exchange_rates_clean
}.items():
    
    print(name)
    print(df.columns.tolist())
    print()

In [ ]:
def clean_text_columns(df):
    
    df = df.copy()
    
    text_columns = df.select_dtypes(
        include=["object","string"]
    ).columns
    
    for col in text_columns:
        
        df[col] = (
            df[col]
            .astype('string')
            .str.strip()
        )
        
        df[col] = (
            df[col]
            .replace({
                '': pd.NA,
                'NA': pd.NA,
                'N/A': pd.NA,
                'NULL': pd.NA,
                'null': pd.NA,
                'None': pd.NA
            })
        )
    
    return df

In [ ]:
customers_clean = clean_text_columns(customers_clean)
products_clean = clean_text_columns(products_clean)
sales_clean = clean_text_columns(sales_clean)
stores_clean = clean_text_columns(stores_clean)
exchange_rates_clean = clean_text_columns(exchange_rates_clean)

In [ ]:
numeric_columns = {
    'Customers': [
        'CustomerKey',
        'Zip Code'
    ],
    'Products': [
        'ProductKey',
        'Unit Cost USD',
        'Unit Price USD',
        'SubcategoryKey',
        'CategoryKey'
    ],
    'Sales': [
        'Line Item',
        'CustomerKey',
        'StoreKey',
        'ProductKey',
        'Quantity'
    ],
    'Stores': [
        'StoreKey',
        'Square Meters'
    ],
    'Exchange Rates': [
        'Exchange'
    ]
}

currency_columns = {
    'Products': [
        'Unit Cost USD',
        'Unit Price USD'
    ]
}

In [ ]:
clean_tables = {
    'Customers': customers_clean,
    'Products': products_clean,
    'Sales': sales_clean,
    'Stores': stores_clean,
    'Exchange Rates': exchange_rates_clean
}

for table_name, columns in numeric_columns.items():
    df = clean_tables[table_name]

    for col in columns:
        if col not in df.columns:
            continue

        values = df[col]

        if col in currency_columns.get(table_name, []):
            values = (
                values
                .astype('string')
                .str.replace('$', '', regex=False)
                .str.replace(',', '', regex=False)
                .str.strip()
            )

        df[col] = pd.to_numeric(values, errors='coerce')

print("Product price/cost data types:")
print(products_clean[['Unit Price USD', 'Unit Cost USD']].dtypes)

print("\nMissing product price/cost values after conversion:")
print(products_clean[['Unit Price USD', 'Unit Cost USD']].isna().sum())

display(
    products_clean[
        ['ProductKey', 'Product Name', 'Unit Price USD', 'Unit Cost USD']
    ].head()
)

In [ ]:
date_columns = {
    
    'Customers': ['Birthday'],
    
    'Sales': [
        'Order Date',
        'Delivery Date'
    ],
    
    'Stores': ['Open Date'],
    
    'Exchange Rates': ['Date']
}
for table_name, columns in date_columns.items():
    
    df = clean_tables[table_name]
    
    for col in columns:
        
        if col in df.columns:
            
            df[col] = pd.to_datetime(
                df[col],
                errors='coerce'
            )

In [ ]:
for name, df in clean_tables.items():
    
    duplicate_count = df.duplicated().sum()
    
    print(
        f"{name}: "
        f"{duplicate_count:,} duplicate rows"
    )
for name in clean_tables:
    
    clean_tables[name] = (
        clean_tables[name]
        .drop_duplicates()
        .reset_index(drop=True)
)    

customers_clean = clean_tables['Customers']
products_clean = clean_tables['Products']
sales_clean = clean_tables['Sales']
stores_clean = clean_tables['Stores']
exchange_rates_clean = clean_tables['Exchange Rates']

## 7. Data integration and analytical dataset

The sales fact table is enriched with product, customer, and store attributes. Join validation helps protect transaction-level granularity and avoid accidental row multiplication.

In [ ]:
key_checks = {
    'Customers': 'CustomerKey',
    'Products': 'ProductKey',
    'Stores': 'StoreKey'
}

for table_name, key in key_checks.items():
    
    df = clean_tables[table_name]
    
    print(
        table_name,
        '|',
        'Rows:', len(df),
        '|',
        'Unique Keys:', df[key].nunique(),
        '|',
        'Missing Keys:', df[key].isna().sum()
    )

In [ ]:
sales_analysis = sales_clean.merge(
    products_clean[
        [
            'ProductKey',
            'Product Name',
            'Brand',
            'Color',
            'Unit Cost USD',
            'Unit Price USD',
            'SubcategoryKey',
            'Subcategory',
            'CategoryKey',
            'Category'
        ]
    ],
    on='ProductKey',
    how='left',
    validate='many_to_one'
)

In [ ]:
sales_analysis = sales_analysis.merge(
    customers_clean[
        [
            'CustomerKey',
            'Gender',
            'Name',
            'City',
            'State Code',
            'State',
            'Zip Code',
            'Country',
            'Continent',
            'Birthday'
        ]
    ],
    on='CustomerKey',
    how='left',
    validate='many_to_one'
)

In [ ]:
sales_analysis = sales_analysis.merge(
    stores_clean[
        [
            'StoreKey',
            'Country',
            'State',
            'Square Meters',
            'Open Date'
        ]
    ],
    on='StoreKey',
    how='left',
    validate='many_to_one',
    suffixes=('', '_Store')
)

In [ ]:
print("Sales rows:", len(sales_clean))
print("Analytical rows:", len(sales_analysis))

print(
    "Sales columns:",
    sales_analysis.shape[1]
)

display(
    sales_analysis.head()
)

## 8. Feature engineering

Transaction-level features include revenue, cost, profit, profit margin, calendar attributes, delivery duration, customer age bands, sales channel, and customer-level RFM measures.

In [ ]:
sales_analysis['Revenue USD'] = (
    sales_analysis['Quantity'] * sales_analysis['Unit Price USD']
)

sales_analysis['Cost USD'] = (
    sales_analysis['Quantity'] * sales_analysis['Unit Cost USD']
)

sales_analysis['Profit USD'] = (
    sales_analysis['Revenue USD'] - sales_analysis['Cost USD']
)

revenue = sales_analysis['Revenue USD']
profit = sales_analysis['Profit USD']

sales_analysis['Profit Margin %'] = (
    profit.div(revenue)
    .mul(100)
    .where(revenue.notna() & revenue.ne(0))
)

print("Financial column data types:")
print(
    sales_analysis[
        ['Quantity', 'Unit Price USD', 'Unit Cost USD',
         'Revenue USD', 'Cost USD', 'Profit USD', 'Profit Margin %']
    ].dtypes
)

print("\nMissing financial values:")
print(
    sales_analysis[
        ['Unit Price USD', 'Unit Cost USD',
         'Revenue USD', 'Cost USD', 'Profit USD']
    ].isna().sum()
)

print("\nFinancial totals:")
print(f"Revenue: ${sales_analysis['Revenue USD'].sum():,.2f}")
print(f"Cost:    ${sales_analysis['Cost USD'].sum():,.2f}")
print(f"Profit:  ${sales_analysis['Profit USD'].sum():,.2f}")

Calendar and delivery features support trend analysis, delivery-duration measurement, and fulfilment performance classification.

In [ ]:
sales_analysis['Order Year'] = (
    sales_analysis['Order Date'].dt.year
)

sales_analysis['Order Month'] = (
    sales_analysis['Order Date'].dt.month
)

sales_analysis['Order Month Name'] = (
    sales_analysis['Order Date']
    .dt.strftime('%b')
)

sales_analysis['Order Quarter'] = (
    sales_analysis['Order Date']
    .dt.to_period('Q')
    .astype(str)
)

sales_analysis['Order Year Month'] = (
    sales_analysis['Order Date']
    .dt.to_period('M')
)

In [ ]:
sales_analysis['Delivery Days'] = (
    sales_analysis['Delivery Date']
    -
    sales_analysis['Order Date']
).dt.days

sales_analysis['Delivery Status'] = np.select(
    
    [
        sales_analysis['Delivery Days'].isna(),
        
        sales_analysis['Delivery Days'] < 0,
        
        sales_analysis['Delivery Days'] <= 2,
        
        sales_analysis['Delivery Days'] <= 5
    ],
    
    [
        'Missing Delivery Date',
        'Invalid',
        'Fast',
        'Standard'
    ],
    
    default='Slow'
)

In [ ]:
analysis_date = (
    sales_analysis['Order Date']
    .max()
)

print(
    "Analysis date:",
    analysis_date
)

sales_analysis['Customer Age'] = (
    (
        sales_analysis['Order Date']
        -
        sales_analysis['Birthday']
    ).dt.days
    /
    365.25
)

sales_analysis['Customer Age'] = (
    sales_analysis['Customer Age']
    .round(1)
)

In [ ]:
sales_analysis['Age Band'] = pd.cut(
    sales_analysis['Customer Age'],
    
    bins=[
        0,
        18,
        25,
        35,
        45,
        55,
        65,
        100
    ],
    
    labels=[
        '<18',
        '18-24',
        '25-34',
        '35-44',
        '45-54',
        '55-64',
        '65+'
    ],
    
    right=False
)

In [ ]:
sales_clean['StoreKey'].min(), sales_clean['StoreKey'].max()
sales_analysis['Channel'] = np.where(
    sales_analysis['StoreKey'] == 0,
    'Online',
    'Store'
)

sales_analysis['Channel'].value_counts()

In [ ]:
customer_rfm = (
    sales_analysis
    .groupby('CustomerKey')
    .agg(
        Recency=(
            'Order Date',
            lambda x: (
                analysis_date - x.max()
            ).days
        ),
        
        Frequency=(
            'Order Number',
            'nunique'
        ),
        
        Monetary=(
            'Revenue USD',
            'sum'
        ),
        
        Profit=(
            'Profit USD',
            'sum'
        )
    )
    .reset_index()
)

In [ ]:
customer_rfm['R_Score'] = pd.qcut(
    customer_rfm['Recency']
    .rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

customer_rfm['F_Score'] = pd.qcut(
    customer_rfm['Frequency']
    .rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_rfm['M_Score'] = pd.qcut(
    customer_rfm['Monetary']
    .rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_rfm['RFM_Score'] = (
    customer_rfm['R_Score']
    +
    customer_rfm['F_Score']
    +
    customer_rfm['M_Score']
)

In [ ]:
def rfm_segment(row):
    
    r = row['R_Score']
    f = row['F_Score']
    m = row['M_Score']
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    
    elif r >= 4 and f >= 3:
        return 'Loyal Customers'
    
    elif r >= 4 and m >= 4:
        return 'High Value Recent'
    
    elif r <= 2 and m >= 4:
        return 'High Value At Risk'
    
    elif r <= 2 and f <= 2:
        return 'Low Engagement'
    
    else:
        return 'Potential Loyalists'

customer_rfm['Segment'] = (
    customer_rfm
    .apply(rfm_segment, axis=1)
)

## 9. Exploratory analysis and KPI development

In [ ]:
total_revenue = (
    sales_analysis['Revenue USD'].sum()
)

total_cost = (
    sales_analysis['Cost USD'].sum()
)

total_profit = (
    sales_analysis['Profit USD'].sum()
)

total_orders = (
    sales_analysis['Order Number']
    .nunique()
)

total_customers = (
    sales_analysis['CustomerKey']
    .nunique()
)

total_units = (
    sales_analysis['Quantity'].sum()
)

profit_margin = (
    total_profit / total_revenue * 100
    if pd.notna(total_revenue) and total_revenue != 0
    else np.nan
)

kpis = pd.Series({
    'Total Revenue': total_revenue,
    'Total Cost': total_cost,
    'Total Profit': total_profit,
    'Profit Margin %': profit_margin,
    'Total Orders': total_orders,
    'Total Customers': total_customers,
    'Units Sold': total_units
})

display(kpis)

In [ ]:
monthly_sales = (
    sales_analysis
    .groupby('Order Year Month')
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
        Customers=('CustomerKey', 'nunique')
    )
    .reset_index()
)

monthly_sales['Revenue MoM %'] = (
    monthly_sales['Revenue']
    .pct_change()
    * 100
)

category_analysis = (
    sales_analysis
    .groupby('Category')
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Units=('Quantity', 'sum'),
        Orders=('Order Number', 'nunique'),
        Customers=('CustomerKey', 'nunique')
    )
    .reset_index()
)

category_analysis['Profit Margin %'] = (
    category_analysis['Profit']
    .div(category_analysis['Revenue'])
    .mul(100)
    .where(
        category_analysis['Revenue'].notna()
        & category_analysis['Revenue'].ne(0)
    )
)

category_analysis = (
    category_analysis
    .sort_values(
        'Revenue',
        ascending=False
    )
)

In [ ]:
product_analysis = (
    sales_analysis
    .groupby(
        [
            'ProductKey',
            'Product Name',
            'Brand',
            'Category'
        ]
    )
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Units=('Quantity', 'sum'),
        Orders=('Order Number', 'nunique')
    )
    .reset_index()
)

product_analysis['Profit Margin %'] = (
    product_analysis['Profit']
    .div(product_analysis['Revenue'])
    .mul(100)
    .where(
        product_analysis['Revenue'].notna()
        & product_analysis['Revenue'].ne(0)
    )
)

top_products = (
    product_analysis
    .sort_values(
        'Revenue',
        ascending=False
    )
    .head(10)
)

display(top_products)

In [ ]:
customer_analysis = (sales_analysis
    .groupby(
        [
            'CustomerKey',
            'Name',
            'Country',
            'State'
        ]
    )
        
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
        Units=('Quantity', 'sum')
    )
    .reset_index()
)

customer_analysis['Average Order Value'] = (
    customer_analysis['Revenue']
    /
    customer_analysis['Orders']
)

repeat_customer_summary = (
    customer_analysis['Orders']
    .value_counts()
    .sort_index()
)
repeat_customers = customer_analysis[
    customer_analysis['Orders'] > 1
]

display(repeat_customer_summary)
repeat_customer_rate = (
    len(repeat_customers)
    /
    len(customer_analysis)
    *
    100
)

print(
    f"Repeat customer rate: "
    f"{repeat_customer_rate:.2f}%"
)

In [ ]:
customer_pareto = (
    customer_analysis
    .sort_values(
        'Revenue',
        ascending=False
    )
    .copy()
)

customer_pareto['Revenue %'] = (
    customer_pareto['Revenue']
    /
    customer_pareto['Revenue'].sum()
    *
    100
)

customer_pareto['Cumulative Revenue %'] = (
    customer_pareto['Revenue %']
    .cumsum()
)

top_80_customers = customer_pareto[
    customer_pareto['Cumulative Revenue %'] <= 80
]

## 10. Business analysis

In [ ]:
monthly_sales[
    [
        'Order Year Month',
        'Revenue',
        'Profit',
        'Revenue MoM %'
    ]
].tail(12)

In [ ]:
category_analysis[
    [
        'Category',
        'Revenue',
        'Profit',
        'Profit Margin %'
    ]
].sort_values(
    'Revenue',
    ascending=False
)

In [ ]:
category_analysis[
    [
        'Category',
        'Revenue',
        'Profit',
        'Profit Margin %'
    ]
].sort_values(
    'Profit Margin %'
)

In [ ]:
top_products[
    [
        'Product Name',
        'Brand',
        'Category',
        'Revenue',
        'Profit',
        'Profit Margin %'
    ]
]

In [ ]:
high_revenue_low_margin = (
    product_analysis[
        (product_analysis['Revenue'] >=
         product_analysis['Revenue'].quantile(0.75))
        &
        (product_analysis['Profit Margin %'] <
         product_analysis['Profit Margin %'].median())
    ]
    .sort_values(
        'Revenue',
        ascending=False
    )
)

display(high_revenue_low_margin)

In [ ]:
customer_pareto.head(20)[
    [
        'CustomerKey',
        'Name',
        'Revenue',
        'Cumulative Revenue %'
    ]
]

In [ ]:
channel_analysis = (
    sales_analysis
    .groupby('Channel')
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
        Customers=('CustomerKey', 'nunique'),
        Units=('Quantity', 'sum')
    )
    .reset_index()
)

channel_analysis['Profit Margin %'] = (
    channel_analysis['Profit']
    .div(channel_analysis['Revenue'])
    .mul(100)
    .where(
        channel_analysis['Revenue'].notna()
        & channel_analysis['Revenue'].ne(0)
    )
)

display(channel_analysis)

In [ ]:
store_analysis = (
    sales_analysis[
        sales_analysis['Channel'] == 'Store'
    ]
    .groupby(
        [
            'StoreKey',
            'Country_Store',
            'State_Store'
        ]
    )
    .agg(
        Revenue=('Revenue USD', 'sum'),
        Profit=('Profit USD', 'sum'),
        Orders=('Order Number', 'nunique'),
        Units=('Quantity', 'sum'),
        Store_Size=('Square Meters', 'first')
    )
    .reset_index()
)

store_analysis['Revenue per Sq Meter'] = (
    store_analysis['Revenue']
    /
    store_analysis['Store_Size']
)

In [ ]:
delivery_analysis = (
    sales_analysis
    .groupby('Delivery Status')
    .agg(
        Orders=('Order Number', 'nunique'),
        Average_Delivery_Days=(
            'Delivery Days',
            'mean'
        ),
        Revenue=('Revenue USD', 'sum')
    )
    .reset_index()
)

display(delivery_analysis)

In [ ]:
rfm_analysis = (
    customer_rfm
    .groupby('Segment')
    .agg(
        Customers=('CustomerKey', 'count'),
        Revenue=('Monetary', 'sum'),
        Profit=('Profit', 'sum'),
        Avg_Recency=('Recency', 'mean'),
        Avg_Frequency=('Frequency', 'mean')
    )
    .reset_index()
)

display(rfm_analysis)

## 11. Visual analytics

The charts summarize sales trends, category and product performance, RFM segments, delivery performance, and the relationship between product revenue and profit margin.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_sales['Order Year Month'].astype(str),
    monthly_sales['Revenue']
)

plt.title(
    'Monthly Revenue Trend'
)

plt.xlabel('Month')

plt.ylabel('Revenue (USD)')

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()

In [ ]:
category_plot = (
    category_analysis
    .sort_values(
        'Revenue',
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

plt.barh(
    category_plot['Category'],
    category_plot['Revenue']
)

plt.title(
    'Revenue by Category'
)

plt.xlabel('Revenue (USD)')

plt.ylabel('Category')

plt.tight_layout()

plt.show()

In [ ]:
plot_data = category_analysis.sort_values(
    'Revenue',
    ascending=False
)

plt.figure(figsize=(12, 6))

x = np.arange(
    len(plot_data)
)

width = 0.35

plt.bar(
    x - width / 2,
    plot_data['Revenue'],
    width,
    label='Revenue'
)

plt.bar(
    x + width / 2,
    plot_data['Profit'],
    width,
    label='Profit'
)

plt.xticks(
    x,
    plot_data['Category'],
    rotation=45,
    ha='right'
)

plt.title(
    'Revenue vs Profit by Category'
)

plt.ylabel('USD')

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
top10 = (
    product_analysis
    .nlargest(
        10,
        'Revenue'
    )
    .sort_values(
        'Revenue'
    )
)

plt.figure(figsize=(12, 7))

plt.barh(
    top10['Product Name'],
    top10['Revenue']
)

plt.title(
    'Top 10 Products by Revenue'
)

plt.xlabel('Revenue (USD)')

plt.tight_layout()

plt.show()

In [ ]:
rfm_plot = (
    rfm_analysis
    .sort_values(
        'Revenue',
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

plt.barh(
    rfm_plot['Segment'],
    rfm_plot['Revenue']
)

plt.title(
    'Revenue Contribution by RFM Segment'
)

plt.xlabel('Revenue (USD)')

plt.tight_layout()

plt.show()

In [ ]:
delivery_plot = (
    delivery_analysis
    .sort_values(
        'Average_Delivery_Days'
    )
)

plt.figure(figsize=(10, 6))

plt.bar(
    delivery_plot['Delivery Status'],
    delivery_plot['Average_Delivery_Days']
)

plt.title(
    'Average Delivery Time by Delivery Status'
)

plt.xlabel('Delivery Status')

plt.ylabel('Average Delivery Days')

plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.scatter(
    product_analysis['Revenue'],
    product_analysis['Profit Margin %'],
    alpha=0.6
)

plt.title(
    'Product Revenue vs Profit Margin'
)

plt.xlabel('Revenue (USD)')

plt.ylabel('Profit Margin (%)')

plt.tight_layout()

plt.show()

## 12. Executive findings and recommendations

This section translates the calculated metrics into business findings and recommended actions. Findings are descriptive and should be treated as signals for investigation, not proof of causation.

In [ ]:
top_category = (
    category_analysis.loc[category_analysis['Revenue'].idxmax()]
    if not category_analysis.empty
    and category_analysis['Revenue'].notna().any()
    else None
)

top_product = (
    product_analysis.loc[product_analysis['Revenue'].idxmax()]
    if not product_analysis.empty
    and product_analysis['Revenue'].notna().any()
    else None
)

valid_margin = (
    category_analysis['Profit Margin %']
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

if valid_margin.empty:
    best_margin_category = None
    worst_margin_category = None
    print("Best/worst margin category unavailable: no valid margin values.")
else:
    best_margin_category = category_analysis.loc[valid_margin.idxmax()]
    worst_margin_category = category_analysis.loc[valid_margin.idxmin()]

best_customer = (
    customer_analysis.loc[customer_analysis['Revenue'].idxmax()]
    if not customer_analysis.empty
    and customer_analysis['Revenue'].notna().any()
    else None
)

best_rfm_segment = (
    rfm_analysis.loc[rfm_analysis['Revenue'].idxmax()]
    if not rfm_analysis.empty
    and rfm_analysis['Revenue'].notna().any()
    else None
)

print("Core metrics prepared.")

In [ ]:
print("=" * 70)
print("EXECUTIVE SUMMARY")
print("=" * 70)

print(f"\n1. Revenue: ${total_revenue:,.2f}")
print(f"2. Profit: ${total_profit:,.2f}")
print(
    f"3. Profit Margin: {profit_margin:.2f}%"
    if pd.notna(profit_margin)
    else "3. Profit Margin: Not available"
)
print(f"4. Orders: {total_orders:,}")
print(f"5. Customers: {total_customers:,}")
print(f"6. Units Sold: {total_units:,}")

if top_category is not None:
    print(
        f"\n7. Top Revenue Category: {top_category['Category']} "
        f"(${top_category['Revenue']:,.2f})"
    )
else:
    print("\n7. Top Revenue Category: Not available")

if top_product is not None:
    print(
        f"8. Top Revenue Product: {top_product['Product Name']} "
        f"(${top_product['Revenue']:,.2f})"
    )
else:
    print("8. Top Revenue Product: Not available")

if best_margin_category is not None:
    print(
        f"9. Highest Margin Category: {best_margin_category['Category']} "
        f"({best_margin_category['Profit Margin %']:.2f}%)"
    )
    print(
        f"10. Lowest Margin Category: {worst_margin_category['Category']} "
        f"({worst_margin_category['Profit Margin %']:.2f}%)"
    )
else:
    print("9. Highest Margin Category: Not available")
    print("10. Lowest Margin Category: Not available")

if best_customer is not None:
    print(
        f"11. Highest-Value Customer: {best_customer['Name']} "
        f"(${best_customer['Revenue']:,.2f})"
    )
else:
    print("11. Highest-Value Customer: Not available")

if best_rfm_segment is not None:
    print(
        f"12. Largest RFM Revenue Segment: {best_rfm_segment['Segment']} "
        f"(${best_rfm_segment['Revenue']:,.2f})"
    )
else:
    print("12. Largest RFM Revenue Segment: Not available")

print(f"\n13. Repeat Customer Rate: {repeat_customer_rate:.2f}%")
print("=" * 70)

In [ ]:
insights = []
recommendations = []

if top_category is not None and total_revenue:
    top_category_share = top_category['Revenue'] / total_revenue * 100
    insights.append(
        f"Revenue leader: {top_category['Category']} generated "
        f"${top_category['Revenue']:,.2f}, representing "
        f"{top_category_share:.1f}% of total revenue."
    )
    recommendations.append(
        f"Review the drivers of {top_category['Category']}'s performance and assess "
        "whether successful practices can be applied to other categories."
    )

if top_product is not None:
    insights.append(
        f"Top product by revenue: {top_product['Product Name']} "
        f"({top_product['Brand']}) generated ${top_product['Revenue']:,.2f}."
    )
    recommendations.append(
        "Review top-product availability, inventory continuity, and margin contribution "
        "before prioritizing additional promotion."
    )

if worst_margin_category is not None:
    insights.append(
        f"Lowest-margin category: {worst_margin_category['Category']} with a calculated "
        f"margin of {worst_margin_category['Profit Margin %']:.2f}%."
    )
    recommendations.append(
        f"Investigate pricing, product mix, discounting, and sourcing costs in "
        f"{worst_margin_category['Category']} before making pricing changes."
    )

if total_revenue:
    top_20_share = (
        customer_pareto.head(20)['Revenue'].sum() / total_revenue * 100
    )
    insights.append(
        f"Customer concentration: the top 20 customers account for "
        f"{top_20_share:.1f}% of total revenue."
    )
    recommendations.append(
        "Track revenue concentration and build retention plans for high-value customers "
        "while broadening the active customer base."
    )

insights.append(
    f"Repeat-customer rate: {repeat_customer_rate:.2f}% of customers placed "
    "more than one distinct order in the analysed dataset."
)
recommendations.append(
    "Use repeat-purchase behaviour and RFM segments to tailor loyalty, reactivation, "
    "and cross-sell campaigns."
)

if 'channel_analysis' in globals() and not channel_analysis.empty:
    channel_leader = channel_analysis.sort_values('Revenue', ascending=False).iloc[0]
    insights.append(
        f"Leading channel by revenue: {channel_leader['Channel']} generated "
        f"${channel_leader['Revenue']:,.2f}."
    )
    recommendations.append(
        "Compare channel revenue alongside profit margin, order volume, and customer reach "
        "before allocating additional channel investment."
    )

if 'delivery_analysis' in globals() and not delivery_analysis.empty:
    valid_delivery = delivery_analysis.dropna(subset=['Average_Delivery_Days'])
    if not valid_delivery.empty:
        slowest_delivery = valid_delivery.sort_values(
            'Average_Delivery_Days', ascending=False
        ).iloc[0]
        insights.append(
            f"Delivery review: '{slowest_delivery['Delivery Status']}' has the highest "
            f"average delivery duration among statuses with valid delivery data "
            f"({slowest_delivery['Average_Delivery_Days']:.2f} days)."
        )
        recommendations.append(
            "Investigate delivery exceptions and long-duration orders by geography, store, "
            "and fulfilment process."
        )

print("KEY BUSINESS INSIGHTS")
for number, insight in enumerate(insights, start=1):
    print(f"{number}. {insight}")

print("\nRECOMMENDED BUSINESS ACTIONS")
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")


## 13. Conclusion, limitations, and next steps

### Conclusion

This project builds a reusable retail analytics workflow from raw CSV files to a consolidated transaction-level dataset, business KPIs, customer segmentation, and decision-oriented visualizations. It evaluates performance across time, products, categories, customers, channels, stores, and delivery outcomes.

The findings generated above identify revenue leaders, margin-review opportunities, customer concentration, repeat-purchase behaviour, and areas for operational investigation. Business decisions should consider both the size of each opportunity and the reliability of the underlying data.

### Data and interpretation considerations

- Product price and cost fields are currency-formatted text in the source data and must be normalized before numeric analysis.
- Missing price, cost, date, or key values should be investigated; they should not automatically be interpreted as zero.
- The analysis uses the source's USD unit price and cost fields. Exchange-rate data is loaded for completeness but is not applied to these USD calculations.
- RFM segments are rule-based descriptive groups, not a predictive churn model.
- Observed associations do not establish causation. Pricing, promotion, inventory, customer experience, and fulfilment hypotheses require additional validation.
- Channel classification based on `StoreKey == 0` should only be retained if that convention is confirmed in the source data.

### Potential next steps

1. Publish an interactive Power BI dashboard with revenue, profit, margin, product, channel, store, delivery, and RFM filters.
2. Add cohort and customer-retention analysis over time.
3. Investigate product-level margin drivers and discount/promotion effects if those fields become available.
4. Validate business definitions with stakeholders and automate data-quality checks in a repeatable refresh process.


In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "data_quality_report.csv": data_quality,
    "sales_analysis.csv": sales_analysis,
    "monthly_sales.csv": monthly_sales,
    "category_analysis.csv": category_analysis,
    "product_analysis.csv": product_analysis,
    "customer_analysis.csv": customer_analysis,
    "customer_pareto.csv": customer_pareto,
    "customer_rfm.csv": customer_rfm,
    "rfm_analysis.csv": rfm_analysis,
    "channel_analysis.csv": channel_analysis,
    "store_analysis.csv": store_analysis,
    "delivery_analysis.csv": delivery_analysis
}

for filename, dataframe in exports.items():
    dataframe.to_csv(output_dir / filename, index=False)

print(f"Exported {len(exports)} analytical files to: {output_dir.resolve()}")
